In [2]:
import cv2
import numpy as np
import re
import time
from pathlib import Path
import os
import h5py
from hloc import extract_features, match_features, pairs_from_retrieval
from hloc.utils.io import get_matches, get_keypoints
from my_pkg.tools import sort_key, pixel_to_geo_coordinates, read_pairs, extract_rotation_angle

In [3]:
output_dir = Path("/home/lty/outputs/RealUAV/city2/netvlad")
output_dir.mkdir(exist_ok=True, parents= True)
output_dir_db = output_dir / "db"
output_dir_db.mkdir(exist_ok=True, parents= True)
output_dir_query = output_dir / "query"
output_dir_query.mkdir(exist_ok=True, parents= True)
netvlad_pairs = output_dir / "pairs-query-netvlad20.txt" 
image_dir = Path("/home/lty/datasets/RealUAV/city2/")
img_list = Path("/home/lty/outputs/RealUAV/city2/img_list.txt")
db_list = Path("/home/lty/outputs/RealUAV/city2/tif_list.txt")
query_list = Path("/home/lty/outputs/RealUAV/city2/uav_list.txt")

In [4]:
retrieval_conf = extract_features.confs["netvlad"]
global_descriptors = extract_features.main(retrieval_conf, image_dir, output_dir,image_list=img_list)


[2025/05/08 11:00:25 hloc INFO] Extracting local features with configuration:
{'model': {'name': 'netvlad'},
 'output': 'global-feats-netvlad',
 'preprocessing': {'resize_max': 1024}}
[2025/05/08 11:00:25 hloc.utils.parsers INFO] Imported 496 images from img_list.txt
100%|██████████| 496/496 [00:11<00:00, 43.06it/s]
[2025/05/08 11:00:42 hloc INFO] Finished exporting features.


In [5]:
db_global_descriptors = extract_features.main(retrieval_conf, image_dir, output_dir_db, image_list=db_list)
query_global_descriptors = extract_features.main(retrieval_conf, image_dir, output_dir_query, image_list=query_list)
t1 = time.time()
pairs_from_retrieval.main(
    query_global_descriptors, netvlad_pairs, num_matched=5, db_prefix="tif", query_prefix="uav", db_descriptors=db_global_descriptors
)
t2 = time.time()
print(f"Pairs from retrieval in {t2 - t1:.1f}s.")

[2025/05/08 11:00:59 hloc INFO] Extracting local features with configuration:
{'model': {'name': 'netvlad'},
 'output': 'global-feats-netvlad',
 'preprocessing': {'resize_max': 1024}}
[2025/05/08 11:00:59 hloc.utils.parsers INFO] Imported 168 images from tif_list.txt
100%|██████████| 168/168 [00:04<00:00, 41.22it/s]
[2025/05/08 11:01:08 hloc INFO] Finished exporting features.
[2025/05/08 11:01:08 hloc INFO] Extracting local features with configuration:
{'model': {'name': 'netvlad'},
 'output': 'global-feats-netvlad',
 'preprocessing': {'resize_max': 1024}}
[2025/05/08 11:01:08 hloc.utils.parsers INFO] Imported 328 images from uav_list.txt
100%|██████████| 328/328 [00:07<00:00, 45.48it/s]
[2025/05/08 11:01:20 hloc INFO] Finished exporting features.
[2025/05/08 11:01:20 hloc INFO] Extracting image pairs from a retrieval database.
[2025/05/08 11:01:21 hloc INFO] Found 1640 pairs.


Pairs from retrieval in 0.4s.


In [15]:
# 打开 HDF5 文件
with h5py.File(db_global_descriptors, 'r') as f:
    # 获取所有数据集的名称
    print("文件中的数据集和分组结构：")
    def print_structure(name, obj):
        """递归打印 HDF5 文件的层次结构"""
        if isinstance(obj, h5py.Group):
            print(f"Group: {name}")
        elif isinstance(obj, h5py.Dataset):
            print(f"Dataset: {name} - Shape: {obj.shape} - Type: {obj.dtype}")
    f.visititems(print_structure)

文件中的数据集和分组结构：
Group: seu_tif_m300
Group: seu_tif_m300/10_3000_1500.tif
Dataset: seu_tif_m300/10_3000_1500.tif/global_descriptor - Shape: (4096,) - Type: float16
Dataset: seu_tif_m300/10_3000_1500.tif/image_size - Shape: (2,) - Type: int64
Group: seu_tif_m300/11_4500_1500.tif
Dataset: seu_tif_m300/11_4500_1500.tif/global_descriptor - Shape: (4096,) - Type: float16
Dataset: seu_tif_m300/11_4500_1500.tif/image_size - Shape: (2,) - Type: int64
Group: seu_tif_m300/12_6000_1500.tif
Dataset: seu_tif_m300/12_6000_1500.tif/global_descriptor - Shape: (4096,) - Type: float16
Dataset: seu_tif_m300/12_6000_1500.tif/image_size - Shape: (2,) - Type: int64
Group: seu_tif_m300/13_7500_1500.tif
Dataset: seu_tif_m300/13_7500_1500.tif/global_descriptor - Shape: (4096,) - Type: float16
Dataset: seu_tif_m300/13_7500_1500.tif/image_size - Shape: (2,) - Type: int64
Group: seu_tif_m300/14_9000_1500.tif
Dataset: seu_tif_m300/14_9000_1500.tif/global_descriptor - Shape: (4096,) - Type: float16
Dataset: seu_tif_m3